# 🏷️ Step 2: 로컬 LLM을 활용한 정책 의제 및 메타데이터 추출기
정제 및 분할된 회의록 발언문에서 로컬 LLM(Ollama `gemma4:e2b`, `qwen2.5-coder:7b` 등)을 활용하여
**정책 분야(`policy_category`)**, **핵심 쟁점(`core_issue`)**, **2~4어절 복합 정책 키워드(`keywords`)**, **요약(`summary`)**을 추출하여 JSONL 데이터로 축적합니다.

### 📌 주요 메타데이터 추출 원칙
1. **엄격한 금지 단어(Negative Constraints)**: '사업', '예산', '지원', '조례', '의원' 등 모든 회의록에 범용적으로 등장하는 1단어 명사 엄격 배제
2. **2~4어절 실질 정책 복합명사 강제**: "농지제도 혁신", "지역 소멸 대응", "학령인구 감소", "공공의료 확충" 등 실질 검색 가능한 키워드 추출
3. **체크포인트(이어하기) & 실시간 Flush 지원**: 기존 추출된 `row_id`를 감지하여 중단 후 재실행 시 중복 없이 이어서 안전하게 추가

In [ ]:
import csv
import json
import os
import re
import requests
from pathlib import Path
import pandas as pd

# ---------------------------------------------------------------------------
# 1. 환경 설정 및 Ollama 모델 지정
# ---------------------------------------------------------------------------
COUNCIL_NAME = "강원특별자치도의회"
TARGET_YEAR = "2024"

# 분할 TSV 폴더 또는 단일 TSV 경로
SPLIT_DIR = Path(f"./data/to_text/{TARGET_YEAR}_광역 회의록/split/{COUNCIL_NAME}")
OUTPUT_JSONL = Path(f"./metadata/cleaned_{TARGET_YEAR}_{COUNCIL_NAME}_metadata.jsonl")

# Ollama 설정
OLLAMA_BASE_URL = "http://localhost:11434"
MODEL_NAME = "gemma4:e2b"  # 대안: qwen2.5-coder:7b, huihui_ai/exaone3.5-abliterated

# 표준 정책 분류 8종
POLICY_CATEGORIES = [
    "경제/산업", "농림/수산", "보건/복지", "교육/문화",
    "환경/안전", "도시/교통", "자치/행정", "기타"
]

print(f"입력 분할 폴더: {SPLIT_DIR.resolve()}")
print(f"출력 메타데이터 파일: {OUTPUT_JSONL.resolve()}")

## 2. 시스템 프롬프트 및 견고한 JSON 파서 정의

In [ ]:
SYSTEM_PROMPT = """당신은 대한민국 지방의회 정책 분석 및 메타데이터 구축 전문가입니다.
주어진 지방의회 회의록 발언을 정밀하게 분석하여, 향후 검색 및 필터링 메타데이터로 활용할 수 있도록 JSON 포맷으로만 응답하세요.

[⚠️ 절대 금지 키워드 (Negative Constraints)]
아래와 같이 모든 회의록에 흔히 등장하는 1단어짜리 포괄적 명사, 기관명, 직함은 'keywords'에 절대로 포함하지 마세요:
- 금지 단어: 사업, 예산, 지원, 조례, 조례안, 의견, 업무, 행정, 의회, 의원, 위원, 발언, 질의, 답변, 감사, 생각, 회의, 내용, 부분

[✅ 권장 키워드 형식]
2~4개 단어가 결합된 '구체적이고 실질적인 정책 의제 복합명사'로 추출하세요:
- 좋은 예시: "농지제도 혁신", "지역 소멸 대응", "미래 인구정책", "일자리 산업 육성", "학령인구 감소", "교육복지 강화", "공공의료 확충"

[출력 포맷 (반드시 아래 순수 JSON 형식으로만 응답할 것, 마크다운 코드블록 없이 순수 JSON만 출력)]
{
  "is_substantive": true,
  "policy_category": "경제/산업",
  "core_issue": "농지 규제 완화를 통한 농가 소득 증대 및 농지제도 혁신 방안",
  "keywords": ["농지제도 혁신", "농업 규제 완화", "경자유전 원칙 개편"],
  "summary": "해방 이후 지속된 농지 소유 규제를 완화하여 청년농 유입과 농업 경쟁력을 강화해야 한다고 주장함."
}

※ policy_category 후보: "경제/산업", "농림/수산", "보건/복지", "교육/문화", "환경/안전", "도시/교통", "자치/행정", "기타" 중 정확히 1개 선택.
※ 만약 단순 의전 멘트(개식 선포, 방청객 환영, 단순 이석 보고 등)로 실질적 정책 논의가 없다면 is_substantive를 false로 하고 keywords는 []로 비워두세요.
"""

def clean_and_parse_json(raw_text: str):
    """LLM 출력 문자열에서 유연하고 안전하게 JSON을 파싱"""
    if not raw_text:
        return None
    text = raw_text.strip()
    if "```" in text:
        text = re.sub(r'```(?:json)?\s*', '', text).replace('```', '').strip()
    m = re.search(r'(\{[\s\S]*\})', text)
    candidate = m.group(1).strip() if m else text
    try:
        return json.loads(candidate)
    except Exception:
        pass
    try:
        fixed = re.sub(r',\s*\}', '}', candidate)
        fixed = re.sub(r',\s*\]', ']', fixed)
        return json.loads(fixed)
    except Exception:
        return None

def call_ollama(prompt: str, model: str = MODEL_NAME) -> str | None:
    """Ollama REST API 호출"""
    try:
        resp = requests.post(
            f"{OLLAMA_BASE_URL}/api/chat",
            json={
                "model": model,
                "messages": [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": prompt}
                ],
                "stream": False,
                "options": {"temperature": 0.1, "top_p": 0.9}
            },
            timeout=60
        )
        if resp.status_code == 200:
            return resp.json().get("message", {}).get("content", "").strip()
    except Exception as e:
        print(f"Ollama 호출 경고: {e}")
    return None

## 3. 체크포인트 기반 메타데이터 추출 실행

In [ ]:
OUTPUT_JSONL.parent.mkdir(parents=True, exist_ok=True)

# 기존 체크포인트 로드 (이미 처리된 row_id 건너뛰기)
processed_ids = set()
if OUTPUT_JSONL.exists():
    with OUTPUT_JSONL.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                try:
                    rec = json.loads(line)
                    if "row_id" in rec:
                        processed_ids.add(int(rec["row_id"]))
                except Exception:
                    pass

print(f"기존 처리된 체크포인트: {len(processed_ids):,}건 감지")

# 입력 분할 TSV 탐색
tsv_files = sorted(SPLIT_DIR.glob("part_*.tsv"))
if not tsv_files:
    tsv_files = sorted(SPLIT_DIR.glob("*.tsv")) + sorted(SPLIT_DIR.glob("*.txt"))

print(f"처리 대상 TSV 파일 수: {len(tsv_files)}개")

extracted_count = 0
MAX_SAMPLE = 5  # 대화형 노트북 테스트를 위해 처음 5개 발언 추출

with OUTPUT_JSONL.open("a", encoding="utf-8") as out_f:
    for tf in tsv_files:
        with tf.open("r", encoding="utf-8-sig", newline="") as in_f:
            reader = csv.DictReader(in_f, delimiter="\t")
            for row in reader:
                r_id = int(row.get("row_id", 0))
                if r_id in processed_ids:
                    continue
                utterance = row.get("utterance", "").strip()
                if len(utterance) < 15:
                    continue
                    
                speaker = f"{row.get('speaker_role','')} {row.get('speaker_name','')} (안건: {row.get('agenda_title','')})"
                prompt = f"[발언자 정보]: {speaker}\n[발언 내용]:\n{utterance[:1500]}"
                
                llm_out = call_ollama(prompt)
                parsed = clean_and_parse_json(llm_out) if llm_out else None
                
                if not parsed:
                    parsed = {
                        "is_substantive": len(utterance) > 50,
                        "policy_category": "기타",
                        "core_issue": row.get("agenda_title", "")[:60],
                        "keywords": [],
                        "summary": utterance[:100] + "..."
                    }
                    
                record = {
                    "row_id": r_id,
                    "date": row.get("date") or row.get("MTG_DE", ""),
                    "assembly": row.get("assembly") or row.get("RASMBLY_NM", COUNCIL_NAME),
                    "speaker_name": row.get("speaker_name", ""),
                    "speaker_role": row.get("speaker_role", ""),
                    "agenda_title": row.get("agenda_title", ""),
                    "utterance": utterance,
                    "is_substantive": parsed.get("is_substantive", True),
                    "policy_category": parsed.get("policy_category", "기타"),
                    "core_issue": parsed.get("core_issue", ""),
                    "keywords": parsed.get("keywords", []),
                    "summary": parsed.get("summary", "")
                }
                
                out_f.write(json.dumps(record, ensure_ascii=False) + "\n")
                out_f.flush()
                processed_ids.add(r_id)
                extracted_count += 1
                
                print(f"✓ [{extracted_count}/{MAX_SAMPLE}] row_id={r_id} | 분야: {record['policy_category']} | 핵심이슈: {record['core_issue'][:30]}...")
                if extracted_count >= MAX_SAMPLE:
                    break
        if extracted_count >= MAX_SAMPLE:
            break

print(f"\n추출 완료! 누적 저장 위치: {OUTPUT_JSONL}")

## 4. 추출된 정책 메타데이터 확인 및 통계

In [ ]:
if OUTPUT_JSONL.exists() and os.path.getsize(OUTPUT_JSONL) > 0:
    records = []
    with OUTPUT_JSONL.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    df_meta = pd.DataFrame(records)
    print(f"총 추출된 메타데이터 수: {len(df_meta)}건")
    display(df_meta[['row_id', 'assembly', 'speaker_name', 'policy_category', 'core_issue', 'keywords']].head(10))
    
    print("\n[정책 분야별 발언 분포]")
    print(df_meta['policy_category'].value_counts())
else:
    print("메타데이터 파일이 비어 있습니다.")